# Smoke train: 100k-step PPO from scratch

Shows the training loop end-to-end on a tiny budget so the pipeline can
be validated without spinning up a 100M-step run:

1. Launch a 100 000-step PPO training on `basesWorkers16x16A` with 8 CPU
   envs against `RandomBiasedAI` only.
2. The trainer writes to `outputs/runs/train-smoke_s1/`.
3. Once done, load the produced agent and play one game.

End-to-end: 3-5 minutes on a 2024-era CPU laptop. Same script the SLURM
scripts under `experiments/single-map/` use, just with smaller numbers.

**Prereqs**: `bash setup/local.sh` once. Run from the repo root.

## 1. Sanity check

In [ ]:
import shutil

from microrts_agent.paths import PROJECT_ROOT

bridge_jar = PROJECT_ROOT / "microrts_agent" / "microrts" / "lib" / "bridge.jar"
assert bridge_jar.exists(), (
    "FAIL: bridge.jar missing. Run bash microrts_agent/microrts/build_bridge.sh"
)
assert shutil.which("java"), "FAIL: Java not on PATH (need JDK 17+)"
print("Prereqs OK")
print("Repo root :", PROJECT_ROOT)

## 2. Launch the training run

100 000 steps, GridNet architecture (smallest at 0.84 M params), no
self-play, no in-training eval. The trainer writes its log to stdout
and saves `agent.pt` + `config.json` + `train.log` under
`outputs/runs/train-smoke_s1/`.

In [ ]:
import subprocess
import sys

run_dir = PROJECT_ROOT / "outputs" / "runs" / "train-smoke_s1"
if run_dir.exists():
    print(f"Note: {run_dir} already exists. Delete it to re-run from scratch.")

cmd = [
    sys.executable,
    "-m",
    "microrts_agent",
    "train",
    "--exp-name",
    "train-smoke_s1",
    "--architecture",
    "gridnet",
    "--map",
    "maps/open_competition/basesWorkers16x16A.xml",
    "--total-timesteps",
    "100000",
    "--num-bot-envs",
    "8",
    "--num-selfplay-envs",
    "0",
    "--num-steps",
    "256",
    "--with-eval",
    "False",
    "--seed",
    "1",
]
print("Running:", " ".join(cmd))
print("(expect 3-5 minutes on CPU)")
result = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True, timeout=900)
print("--- last 1500 chars of stdout ---")
print(result.stdout[-1500:])
if result.returncode != 0:
    print("--- stderr ---")
    print(result.stderr[-1500:])

## 3. Inspect what was produced

In [ ]:
for path in sorted(run_dir.iterdir()):
    size = path.stat().st_size if path.is_file() else None
    size_str = f"{size:>10,} bytes" if size is not None else "          <dir>"
    print(f"  {size_str}  {path.name}")

## 4. Load the just-trained agent and play one game

Same `evaluate` flow as `examples/load_agent.ipynb`, but against the
agent we just trained instead of a shipped one. 100k steps is far
below the budget needed for real performance: expect lots of losses
against the scripted bots.

In [ ]:
eval_cmd = [
    sys.executable,
    "-m",
    "microrts_agent",
    "evaluate",
    "--agent",
    str(run_dir),
    "--opponent",
    "RandomBiasedAI",
    "--maps",
    "maps/open_competition/basesWorkers16x16A.xml",
    "--nb_games",
    "2",
    "--max-steps",
    "2000",
]
result = subprocess.run(
    eval_cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True, timeout=300
)
print(result.stdout[-1500:])
if result.returncode != 0:
    print("--- stderr ---")
    print(result.stderr[-1500:])

## Next steps

- Bigger budget: bump `--total-timesteps` to 1_000_000 (10 minutes
  on CPU) or 100_000_000 (~10 hours, needs GPU).
- Stronger architecture: switch `--architecture gridnet` to
  `unet_entity_cbam_deep` (the architecture used by `UECD-SingleMap-Best`).
- Production-grade flags: copy any of the SLURM scripts under
  `experiments/single-map/` for the full ablation-positive feature
  stack.
- See `examples/analyze_training_log.ipynb` to plot the win-rate curve
  of this run (point the parser at `train-smoke_s1`).